In [ ]:
!pip install transformers datasets pandas scikit-learn torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# --- Monta Google Drive (Colab) ---
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
base_path = '/content/drive/MyDrive/ModelloStanceDetection'

In [ ]:
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification, TrainingArguments, Trainer
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import StratifiedKFold
import numpy as np

In [ ]:
# --- Dataset PyTorch ---

class StanceDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
stance_mapping = {
    0: "NOT DEFINED",
    1: "Promotional",
    2: "Neutral",
    3: "Discouraging",
    4: "Ambiguous",
    5: "indefinite/ironic",
}

In [ ]:
def load_and_preprocess_data(filepath='merged_data_pulsar_groundtruth.csv'):
    df = pd.read_csv(filepath, index_col=None)
    df['source'] = df['source'].astype(str)
    df = df[~df.stance.isin([0, 5])]
    df['stance_name'] = df['stance'].apply(lambda x: stance_mapping[x])
    df_for_classification = df[df['source'].str.contains("Facebook Pages") | (
            df['source'].str.contains("X") & df['post subtype'].str.contains("original post"))]

    # Estrazione Mese e Anno da 'folder' (opzionale, mantenuto per completezza)
    def extract_year_month(folder_name):
        match = re.search(r"(\w+) (\d{4})", folder_name)
        if match:
            month_name = match.group(1)
            year = match.group(2)
            month_mapping = {
                "gennaio": "01", "febbraio": "02", "marzo": "03", "aprile": "04",
                "maggio": "05", "giugno": "06", "luglio": "07", "agosto": "08",
                "settembre": "09", "ottobre": "10", "novembre": "11", "dicembre": "12"
            }
            month_number = month_mapping.get(month_name.lower())
            if month_number:
                return f"{year}_{month_number}"
        return "unknown_date"

    df_for_classification['year_month'] = df_for_classification['folder'].apply(extract_year_month)

    texts = df_for_classification['content'].tolist()
    labels = df_for_classification['stance'].tolist()


    # --- Stratificazione SOLO su 'stance' ---
    strata = df_for_classification['stance']

    return texts, labels, strata

In [ ]:
# --- Caricamento Dati ---
import re
texts, labels, strata = load_and_preprocess_data()

<ipython-input-7-c7d322bfb88b>:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_for_classification['year_month'] = df_for_classification['folder'].apply(extract_year_month)


In [ ]:
len(texts)

2265

In [ ]:
len(labels)

2265

In [ ]:
len(strata)

2265

In [ ]:
# --- Configurazione BERT ---
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [ ]:
# --- Stratified K-Fold ---
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

In [ ]:
# --- Funzione compute_metrics  ---
def compute_metrics(eval_pred, unique_labels, label_mapping):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=-1)
    report = classification_report(labels, preds, target_names=[str(label) for label in unique_labels], output_dict=True)
    return {"f1": report['macro avg']['f1-score']}

In [ ]:
from sklearn.metrics import classification_report
# --- Loop di Cross-Validation ---
fold_reports = []  # Lista per salvare i report di ogni fold
for fold, (train_index, val_index) in enumerate(skf.split(texts, strata)):
    print(f"Fold {fold + 1}/{n_splits}")

    train_texts = [texts[i] for i in train_index]
    val_texts = [texts[i] for i in val_index]
    train_labels = [labels[i] for i in train_index]
    val_labels = [labels[i] for i in val_index]

    unique_train_labels = sorted(list(set(train_labels)))
    label_mapping = {label: i for i, label in enumerate(unique_train_labels)}
    train_labels_remapped = [label_mapping[label] for label in train_labels]
    val_labels_remapped = [label_mapping[label] for label in val_labels]

    num_labels = len(unique_train_labels)
    #print(f"Fold {fold+1} - Unique training labels: {unique_train_labels}, num_labels: {num_labels}")

    train_encodings = tokenizer(train_texts, truncation=True, padding=True)
    val_encodings = tokenizer(val_texts, truncation=True, padding=True)

    train_dataset = StanceDataset(train_encodings, train_labels_remapped)
    val_dataset = StanceDataset(val_encodings, val_labels_remapped)

    model = BertForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

    training_args = TrainingArguments(
        output_dir=f'{base_path}/results/fold_{fold}',
        num_train_epochs=5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=64,
        warmup_steps=500,
        weight_decay=0.01,
        logging_dir=f'{base_path}/logs/fold_{fold}',
        logging_steps=10,
        evaluation_strategy="epoch",
        save_strategy="epoch",  # Salva il modello ad ogni epoca
        load_best_model_at_end=True,  # Carica il miglior modello alla fine
        metric_for_best_model="f1",  # Usa l'F1-score (macro) per scegliere il modello migliore.
        greater_is_better=True # F1 score più alto è meglio.
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=lambda eval_pred: compute_metrics(eval_pred, unique_train_labels, label_mapping)
    )

    trainer.train()

    # --- Predizioni e Classification Report (per fold) ---
    predictions = trainer.predict(val_dataset)
    y_pred = np.argmax(predictions.predictions, axis=-1)
    y_true = val_labels_remapped

    report = classification_report(y_true, y_pred, target_names=[str(label) for label in unique_train_labels], output_dict=True)
    print(f"Classification Report - Fold {fold + 1}:\n")
    print(classification_report(y_true, y_pred, target_names=[str(label) for label in unique_train_labels]))
    fold_reports.append(report)

    # --- Salva  il modello e il tokenizer ---
    model.save_pretrained(f"{base_path}/results/fold_{fold}/best_model")
    tokenizer.save_pretrained(f"{base_path}/results/fold_{fold}/best_model")




Fold 1/5


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss,F1
1,1.092300,1.008754,0.308614
2,0.969800,0.834015,0.439207
3,0.871200,1.004173,0.427293
4,0.773600,0.817901,0.495731
5,0.524900,0.896624,0.495101


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/m

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/m

...

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss,F1
1,1.016200,1.051715,0.354415
2,0.863100,0.974899,0.425808
3,0.804300,0.900173,0.460814
4,0.764400,0.853593,0.460692
5,0.632000,0.905422,0.491596


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/m

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/m

Classification Report - Fold 5:

              precision    recall  f1-score   support

         1.0       0.68      0.74      0.71       209
         2.0       0.54      0.47      0.50       137
         3.0       0.73      0.79      0.76        98
         4.0       0.00      0.00      0.00         9

    accuracy                           0.65       453
   macro avg       0.49      0.50      0.49       453
weighted avg       0.63      0.65      0.64       453



In [ ]:
# --- Riepilogo Finale (Media dei report) ---
print("\n--- Riepilogo Finale (Media dei Classification Report) ---")
# Calcola la media delle metriche su tutti i fold
avg_report = {}
for metric in ['precision', 'recall', 'f1-score', 'support']:
    avg_report[metric] = {}
    for label in fold_reports[0].keys(): #itera sulle chiavi del primo dizionario (che sono le label)
        if label not in ['accuracy', 'macro avg', 'weighted avg']: #ignora accuracy, macro avg e weighted avg perché vanno calcolate a parte
            # Calcola media per la metrica corrente e la label corrente
            avg_report[metric][label] = np.mean([fold_report[label][metric] for fold_report in fold_reports])
        elif label == 'accuracy':
            avg_report[label] = np.mean([fold_report[label] for fold_report in fold_reports])
        elif label in ['macro avg', 'weighted avg']:
            avg_report[label] = {}
            avg_report[label][metric] = np.mean([fold_report[label][metric] for fold_report in fold_reports])

#stampa il report medio
for label, metrics in avg_report.items():
     if isinstance(metrics, dict):
        print(f"{label}:")
        for metric, value in metrics.items():
            print(f"  {metric}: {value:.4f}")
     else:
        print(f"{label}: {metrics:.4f}")


--- Riepilogo Finale (Media dei Classification Report) ---
precision:
  1.0: 0.6721
  2.0: 0.5744
  3.0: 0.7481
  4.0: 0.0000
accuracy: 0.6592
macro avg:
  support: 453.0000
weighted avg:
  support: 453.0000
recall:
  1.0: 0.7354
  2.0: 0.5253
  3.0: 0.7432
  4.0: 0.0000
f1-score:
  1.0: 0.7009
  2.0: 0.5457
  3.0: 0.7453
  4.0: 0.0000
support:
  1.0: 209.4000
  2.0: 137.4000
  3.0: 97.4000
  4.0: 8.8000
